In [ ]:
import pandas as pd
from pymongo import MongoClient
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
mongo_uri = os.getenv("MONGODB_URI")

In [ ]:
DB_NAME = "books_db"
REVIEWS_COLLECTION = "reviews"

client = MongoClient(mongo_uri)
db = client[DB_NAME]

reviews_col = db[REVIEWS_COLLECTION]

print("Connected successfully")

In [ ]:
reviews = list(reviews_col.find({}))

df = pd.DataFrame(reviews)

print(df.shape)
df.head()

In [ ]:
df["review_object_id"] = df["_id"].astype(str)
print(df.columns)
df.head(2)

In [ ]:
df = df[[
    "review_object_id",
    "book_id",
    "review_rating",
    "review_text"
]].copy()

df = df.dropna(subset=["review_text"])
df["review_text"] = df["review_text"].astype(str)

print(df.shape)
df.head()

In [ ]:
!pip install langdetect

In [ ]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 42

In [ ]:
import re

def basic_clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

df["review_text_clean"] = df["review_text"].apply(basic_clean_text)

df[["review_text", "review_text_clean"]].head()

In [ ]:
def detect_language(text):
    try:
        if len(text.strip()) < 20:
            return "unknown"
        return detect(text)
    except:
        return "unknown"

df["language"] = df["review_text_clean"].apply(detect_language)

df["language"].value_counts().head(10)

In [ ]:
df = df[df["language"] == "en"].copy()

print(df.shape)
df[["review_object_id", "book_id", "language", "review_text_clean"]].head()

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    return analyzer.polarity_scores(text)["compound"]

df["sentiment_score"] = df["review_text_clean"].apply(get_sentiment_score)

def get_sentiment_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment_score"].apply(get_sentiment_label)

df[["review_text_clean", "sentiment_score", "sentiment_label"]].head()

In [ ]:
from nrclex import NRCLex

emotion_model = NRCLex()

lexicon = emotion_model.__lexicon__

print(type(lexicon))
print(len(lexicon))

# Show first 5 words in the lexicon
list(lexicon.items())[:5]

In [ ]:
def simple_tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"\b[a-z]+\b", text)
    return tokens

TARGET_EMOTIONS = [
    "joy",
    "anger",
    "sadness",
    "fear",
    "surprise"
]

def get_emotion_features(text):
    tokens = simple_tokenize(text)

    emotion_counts = {
        "joy": 0,
        "anger": 0,
        "sadness": 0,
        "fear": 0,
        "surprise": 0
    }

    total_words = len(tokens)
    emotional_words = 0

    for token in tokens:
        emotions = lexicon.get(token, [])

        if isinstance(emotions, dict):
            emotions = list(emotions.keys())

        has_emotion = False

        for emotion in emotions:
            if emotion in TARGET_EMOTIONS:
                emotion_counts[emotion] += 1
                has_emotion = True

        if has_emotion:
            emotional_words += 1

    total_emotion_hits = sum(emotion_counts.values())

    if total_words == 0 or total_emotion_hits == 0:
        return pd.Series({
            "emotion_intensity": 0,
            "emotion_joy": 0,
            "emotion_anger": 0,
            "emotion_sadness": 0,
            "emotion_fear": 0,
            "emotion_surprise": 0,
            "confidence": 0
        })

    emotion_joy = emotion_counts["joy"] / total_emotion_hits
    emotion_anger = emotion_counts["anger"] / total_emotion_hits
    emotion_sadness = emotion_counts["sadness"] / total_emotion_hits
    emotion_fear = emotion_counts["fear"] / total_emotion_hits
    emotion_surprise = emotion_counts["surprise"] / total_emotion_hits

    # How emotionally dense the review is
    raw_intensity = emotional_words / total_words

    # Normalize intensity.
    # Multiplying by 5 makes normal values easier to compare.
    # min(..., 1) keeps the value between 0 and 1.
    emotion_intensity = min(raw_intensity * 5, 1)

    # How dominant the strongest emotion is
    strongest_emotion_score = max(
        emotion_joy,
        emotion_anger,
        emotion_sadness,
        emotion_fear,
        emotion_surprise
    )

    confidence = strongest_emotion_score

    return pd.Series({
        "emotion_intensity": emotion_intensity,
        "emotion_joy": emotion_joy,
        "emotion_anger": emotion_anger,
        "emotion_sadness": emotion_sadness,
        "emotion_fear": emotion_fear,
        "emotion_surprise": emotion_surprise,
        "confidence": confidence
    })

In [ ]:
sample_text = df["review_text_clean"].iloc[0]

get_emotion_features(sample_text)

In [ ]:
emotion_features = df["review_text_clean"].apply(get_emotion_features)

df = pd.concat([df, emotion_features], axis=1)

df[[
    "review_object_id",
    "book_id",
    "emotion_intensity",
    "emotion_joy",
    "emotion_anger",
    "emotion_sadness",
    "emotion_fear",
    "emotion_surprise",
    "confidence"
]].head()

In [ ]:
nlp_df = df[[
    "review_object_id",
    "book_id",
    "sentiment_score",
    "sentiment_label",
    "emotion_intensity",
    "emotion_joy",
    "emotion_anger",
    "emotion_sadness",
    "emotion_fear",
    "emotion_surprise",
    "confidence"
]].copy()

nlp_df.head()

In [ ]:
NLP_COLLECTION = "review_nlp_analysis"
nlp_col = db[NLP_COLLECTION]

# delete old analysis documents
delete_result = nlp_col.delete_many({})
print("Deleted old documents:", delete_result.deleted_count)

nlp_records = nlp_df.to_dict(orient="records")
nlp_records[:2]

In [ ]:
if len(nlp_records) > 0:
    result = nlp_col.insert_many(nlp_records)
    print("Inserted documents:", len(result.inserted_ids))
else:
    print("No records to insert")

In [ ]:
print("Original reviews:", len(df))
print("NLP records:", len(nlp_df))
print("Saved in MongoDB:", nlp_col.count_documents({}))